Transformation is done based on this document -- [Link](https://docs.google.com/document/d/1_wt-wO18sa5iQBy3b-WpSHDjCqRl3MeN67N1v3dTZtQ/edit#heading=h.usgit3agfa62)

In [1]:
import pandas as pd
from shapely.geometry import Point
import geopandas as gpd
from geopandas import GeoDataFrame

In [4]:
# importing village shapefile from bharatmaps - 
bharat_maps_villages = gpd.read_file(r"D:\CivicDataLab_IDS-DRR\IDS-DRR_Github\Deployment\flood-data-ecosystem-UP\Maps\up_ids-drr_shapefiles\UP_Villages_Boundary_Simplified\UP_Villages_Boundary_Simplified.geojson")

In [3]:
%pip install openpyxl

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
mission_antyodaya_df = pd.read_csv(r'D:\CivicDataLab_IDS-DRR\IDS-DRR_Github\Deployment\flood-data-ecosystem-UP\Sources\ANTYODAYA\data\antyodaya_raw_data.csv')

In [5]:
mission_antyodaya_df.columns.to_list()

['state_code',
 'state_name',
 'state_name_sl',
 'district_code',
 'district_name',
 'district_name_sl',
 'sub_district_code',
 'sub_district_name',
 'sub_district_name_sl',
 'block_code',
 'block_name',
 'block_name_sl',
 'gp_code',
 'gp_name',
 'gp_name_sl',
 'village_code',
 'village_name',
 'village_name_sl',
 'is_gp_village',
 'pc_code',
 'ac_code',
 'total_population',
 'male_population',
 'female_population',
 'total_hhd',
 'total_no_of_elected_representatives',
 'total_no_of_elect_rep_oriented_under_rgsa',
 'total_no_of_elect_rep_undergone_training_under_rgsa',
 'is_benficiearies_list_all_scheme_displayed',
 'is_benficiearies_list_all_scheme_approved',
 'is_list_of_work_displayed',
 'no_of_standing_committees_constitued',
 'no_of_standing_committees_meetings',
 'is_gp_prepared_disaster_plan',
 'is_gp_facilitate_birth_death_cert',
 'is_gp_facilitate_income_cert',
 'is_gp_facilitate_caste_cert',
 'is_gp_facilitate_building_plan_approval',
 'is_gp_facilitate_onlinepay_prop_tax',
 

In [4]:
mission_antyodaya_df.shape

(100136, 255)

In [119]:
mission_antyodaya_geometry = [
        Point(xy) for xy in zip(
            mission_antyodaya_df.village_longitude,
            mission_antyodaya_df.village_latitude
        )
    ]

In [120]:
gdf_points = GeoDataFrame(mission_antyodaya_df, geometry=mission_antyodaya_geometry)

In [128]:
# importing odisha subdistrict shapefile as proxy for revenue circles
gdf_polygons = gpd.read_file(r'D:\CDL\flood-data-ecosystem-UP\Maps\up_ids-drr_shapefiles\UP_Subdistrict_final_modified.geojson')
gdf_polygons = gdf_polygons.to_crs('EPSG:4326')
#gdf_polygons = gdf_polygons.rename(columns = {'SDTCODE11':'sub_district_code'}, inplace = True)

In [ ]:
gdf_points.crs = gdf_polygons.crs
print(gdf_polygons.crs)


EPSG:4326


In [131]:
result = gpd.sjoin(gdf_points, gdf_polygons, how="left", predicate="within")

In [132]:
result.head()

,state_code,state_name,state_name_sl,district_code,district_name,district_name_sl,sub_district_code,sub_district_name,sub_district_name_sl,block_code,...,sdtname,st_area_sh,st_length_,remarks,dist_lgd,state_lgd,subdt_lgd,ac_no,st_length(shape),st_area(shape)
0,9,UTTAR PRADESH,NaN,125,BAHRAICH,NaN,921,Bahraich,NaN,896,...,Kaiserganj,0.0,0.0,,125.0,9.0,922.0,287.0,225300.559651,1.273851e+09
1,9,UTTAR PRADESH,NaN,120,PRAYAGRAJ,NaN,889,Phulpur,NaN,831,...,Phulpur,0.0,0.0,,120.0,9.0,889.0,255.0,226877.385519,9.160798e+08
2,9,UTTAR PRADESH,NaN,168,MAU,NaN,972,Madhuban,NaN,1380,...,Madhuban,0.0,0.0,,168.0,9.0,972.0,353.0,163748.207822,5.868289e+08
3,9,UTTAR PRADESH,NaN,143,FIROZABAD,NaN,772,Firozabad,NaN,1118,...,Firozabad,0.0,0.0,,143.0,9.0,772.0,95.0,229294.467156,5.750214e+08
4,9,UTTAR PRADESH,NaN,134,BULANDSHAHR,NaN,747,Siana,NaN,1025,...,Siana,0.0,0.0,,134.0,9.0,747.0,66.0,220380.501638,8.869488e+08


In [133]:
result.shape

(100136, 273)

In [134]:
empty_sdtname_rows = result['sdtname'].isnull() | (result['sdtname'] == '')

filtered_df = result[empty_sdtname_rows]

filtered_df

,state_code,state_name,state_name_sl,district_code,district_name,district_name_sl,sub_district_code,sub_district_name,sub_district_name_sl,block_code,...,sdtname,st_area_sh,st_length_,remarks,dist_lgd,state_lgd,subdt_lgd,ac_no,st_length(shape),st_area(shape)
362,9,UTTAR PRADESH,NaN,169,MEERUT,NaN,733,Mawana,NaN,1395,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2252,9,UTTAR PRADESH,NaN,132,BIJNOR,NaN,712,Najibabad,NaN,996,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2485,9,UTTAR PRADESH,NaN,165,MAHOBA,NaN,868,Kulpahar,NaN,1357,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2486,9,UTTAR PRADESH,NaN,165,MAHOBA,NaN,868,Kulpahar,NaN,1357,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2487,9,UTTAR PRADESH,NaN,165,MAHOBA,NaN,868,Kulpahar,NaN,1357,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
98784,9,UTTAR PRADESH,NaN,135,CHANDAULI,NaN,993,Chandauli,NaN,1033,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
98790,9,UTTAR PRADESH,NaN,135,CHANDAULI,NaN,993,Chandauli,NaN,1033,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
98991,9,UTTAR PRADESH,NaN,660,SHAMLI,NaN,707,Shamli,NaN,1430,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
99087,9,UTTAR PRADESH,NaN,130,BAREILLY,NaN,784,Baheri,NaN,973,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [135]:
empty_sdtname_rows = result['sub_district_name'].isnull() | (result['sub_district_name'] == '')

filtered_df = result[empty_sdtname_rows]

filtered_df

,state_code,state_name,state_name_sl,district_code,district_name,district_name_sl,sub_district_code,sub_district_name,sub_district_name_sl,block_code,...,sdtname,st_area_sh,st_length_,remarks,dist_lgd,state_lgd,subdt_lgd,ac_no,st_length(shape),st_area(shape)


In [136]:
#renaming sub-district code columns to match
gdf_polygons_renamed = gdf_polygons.rename(columns = {'sdtcode11':'sub_district_code'})
gdf_polygons_renamed = gdf_polygons_renamed.to_crs('EPSG:4326')

gdf_polygons_renamed.head()

,objectid,stcode11,dtcode11,sub_district_code,stname,dtname,sdtname,st_area_sh,st_length_,remarks,dist_lgd,state_lgd,subdt_lgd,ac_no,st_length(shape),st_area(shape),geometry
0,09-168-00865,09,168,00865,UTTAR PRADESH,Hamirpur,Rath,0.0,0.0,,149,9,865,229,205155.793938,9.132342e+08,"MULTIPOLYGON (((79.41425 25.79315, 79.41489 25..."
1,09-200-01005,09,200,01005,UTTAR PRADESH,Sonbhadra,Robertsganj,0.0,0.0,,184,9,1005,401,445254.946921,4.053319e+09,"MULTIPOLYGON (((83.05548 24.95842, 83.05663 24..."
2,09-177-00902,09,177,00902,UTTAR PRADESH,Ayodhya,Rudauli,0.0,0.0,,140,9,902,271,209118.573449,7.294307e+08,"MULTIPOLYGON (((81.7637 26.86081, 81.76371 26...."
3,09-185-00940,09,185,00940,UTTAR PRADESH,Basti,Rudhauli,0.0,0.0,,131,9,940,309,156884.643689,2.995772e+08,"MULTIPOLYGON (((82.78828 27.06198, 82.78839 27..."
4,09-190-00960,09,190,00960,UTTAR PRADESH,Deoria,Rudrapur,0.0,0.0,,137,9,960,336,182005.111691,4.947448e+08,"MULTIPOLYGON (((83.65444 26.58325, 83.65667 26..."


In [137]:
#gdf_polygons and bharat_maps_villages have slightly different sub-district codes. removing unnecessary trailing'0'
gdf_polygons_renamed['sub_district_code'] = gdf_polygons_renamed['sub_district_code'].apply(lambda x: x[1:] if x.startswith('0') else x)
gdf_polygons_renamed

,objectid,stcode11,dtcode11,sub_district_code,stname,dtname,sdtname,st_area_sh,st_length_,remarks,dist_lgd,state_lgd,subdt_lgd,ac_no,st_length(shape),st_area(shape),geometry
0,09-168-00865,09,168,0865,UTTAR PRADESH,Hamirpur,Rath,0.0,0.0,,149,9,865,229,205155.793938,9.132342e+08,"MULTIPOLYGON (((79.41425 25.79315, 79.41489 25..."
1,09-200-01005,09,200,1005,UTTAR PRADESH,Sonbhadra,Robertsganj,0.0,0.0,,184,9,1005,401,445254.946921,4.053319e+09,"MULTIPOLYGON (((83.05548 24.95842, 83.05663 24..."
2,09-177-00902,09,177,0902,UTTAR PRADESH,Ayodhya,Rudauli,0.0,0.0,,140,9,902,271,209118.573449,7.294307e+08,"MULTIPOLYGON (((81.7637 26.86081, 81.76371 26...."
3,09-185-00940,09,185,0940,UTTAR PRADESH,Basti,Rudhauli,0.0,0.0,,131,9,940,309,156884.643689,2.995772e+08,"MULTIPOLYGON (((82.78828 27.06198, 82.78839 27..."
4,09-190-00960,09,190,0960,UTTAR PRADESH,Deoria,Rudrapur,0.0,0.0,,137,9,960,336,182005.111691,4.947448e+08,"MULTIPOLYGON (((83.65444 26.58325, 83.65667 26..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
311,09-139-00737,09,139,0737,UTTAR PRADESH,Baghpat,Khekada,0.0,0.0,,124,9,737,52,99117.973402,2.436437e+08,"MULTIPOLYGON (((77.2851 28.92862, 77.28633 28...."
312,09-137-00718,09,137,0718,UTTAR PRADESH,Amroha,Kanth,0.0,0.0,,154,9,718,25,164751.224731,3.896811e+08,"MULTIPOLYGON (((78.65679 29.14532, 78.65757 29..."
313,09-202-01011,09,202,1011,UTTAR PRADESH,Kasganj,Sahawar,0.0,0.0,,633,9,1011,101,227960.411531,5.795948e+08,"MULTIPOLYGON (((78.85382 27.91837, 78.85485 27..."
314,09-178-00911,09,178,0911,UTTAR PRADESH,Ambedkar Nagar,Bhiti,0.0,0.0,,121,9,911,277,168213.889736,3.854525e+08,"MULTIPOLYGON (((82.35458 26.57984, 82.35516 26..."


In [17]:
#The splitting into tagged+untagged has been skipped as all the points seem to have an associated subdistrict - leaving it in in case this needs to be changed later
#untagged = result[result.sdtname.isnull()]
#tagged = result.dropna(subset=['sdtname'])

In [18]:
#tagged.shape

In [19]:
#untagged.shape

In [138]:
def format_entry(entry):
    if pd.isnull(entry):
        return entry
    entry = entry.lower()              # Convert to lowercase
    entry = entry.replace(' ', '_')    # Replace spaces with underscores
    entry = entry.replace('.', '-')    # Replace periods with hyphens
    return entry

# Apply the function to the specific column
result['sub_district_name'] = result['sub_district_name'].apply(format_entry)

In [139]:
#Original code for revenue circles - not removing as it may be required later
gdf_polygons = gdf_polygons.to_crs('EPSG:4326')
result_bm = bharat_maps_villages[bharat_maps_villages.vilcode11.isin(result.village_code.to_list())]
result_bm['geometry'] = result_bm.geometry.centroid
result_bm = result_bm.to_crs('EPSG:4326')
result_bm = gpd.sjoin(result_bm, gdf_polygons, how="left", predicate="within")

NameError: name 'bharat_maps_villages' is not defined

In [22]:

'''untagged = pd.merge(untagged.drop(['sub_district_code','sdtname'],axis=1),untagged_bm[['VILCODE11','sub_district_code','sdtname']],
        left_on='village_code', right_on='vil_lgd')[untagged.columns]
        '''

"untagged = pd.merge(untagged.drop(['sub_district_code','sdtname'],axis=1),untagged_bm[['VILCODE11','sub_district_code','sdtname']],\n        left_on='village_code', right_on='vil_lgd')[untagged.columns]\n        "

In [23]:
#result = pd.concat([tagged,untagged])

In [24]:
#result.to_csv(r'D:\CivicDataLabs_IDS-DRR\IDS-DRR_Github\flood-data-ecosystem-Odisha\Sources\ANTYODAYA\data/MissionAntyodaya2020_Odisha_untaggedRC.csv')

In [141]:
result[['gp_code','village_code','sub_district_code','sub_district_name','total_hhd',
        'availablility_hours_of_domestic_electricity', 'availability_of_telephone_services',
       'total_hhd_having_piped_water_connection', 'total_hhd_not_having_sanitary_latrines']].to_csv(
    r'D:\CDL\flood-data-ecosystem-UP\Sources\ANTYODAYA\data\UP_ANTYODAYA.csv', index=False)

#net_sown_area_in_hac

In [26]:
#original with revenue circle
''' result[['gp_code','village_code','object_id','revenue_ci','net_sown_area_in_hac','total_hhd',
        'availablility_hours_of_domestic_electricity', 'availability_of_telephone_services',
       'total_hhd_having_piped_water_connection', 'total_hhd_not_having_sanitary_latrines']].to_csv(
    r'D:\CivicDataLabs_IDS-DRR\IDS-DRR_Github\flood-data-ecosystem-Odisha\Sources\ANTYODAYA\data\MissionAntyodaya2020_Odisha_vul.csv', index=False)'''

<>:2: SyntaxWarning: invalid escape sequence '\C'
<>:2: SyntaxWarning: invalid escape sequence '\C'
C:\Users\cdl\AppData\Local\Temp\ipykernel_33052\1810735516.py:2: SyntaxWarning: invalid escape sequence '\C'
  ''' result[['gp_code','village_code','object_id','revenue_ci','net_sown_area_in_hac','total_hhd',


" result[['gp_code','village_code','object_id','revenue_ci','net_sown_area_in_hac','total_hhd',\n        'availablility_hours_of_domestic_electricity', 'availability_of_telephone_services',\n       'total_hhd_having_piped_water_connection', 'total_hhd_not_having_sanitary_latrines']].to_csv(\n    r'D:\\CivicDataLabs_IDS-DRR\\IDS-DRR_Github\x0clood-data-ecosystem-Odisha\\Sources\\ANTYODAYA\\data\\MissionAntyodaya2020_Odisha_vul.csv', index=False)"

# Number of households in each urban area

In [3]:
od_urban_shapes = gpd.read_file(r'D:\CDL\flood-data-ecosystem-UP\Maps\up_ids-drr_shapefiles\up_urban_final.geojson')
od_urban_shapes =  od_urban_shapes.loc[od_urban_shapes['stcode11'] == "09"]
print(od_urban_shapes.crs)


od_urban_shapes

EPSG:4326


,vilnam_soi,stcode11,dtcode11,sdtcode11,stname,dtname,sdtname,vilname11,vil_lgd,dist_lgd,...,gp_code,gp_name,subdt_lgd,block_name,block_lgd,search_village,st_length(shape),st_area(shape),object_id,geometry
0,KALWARI,09,146,00766,UTTAR PRADESH,Agra,Agra,Kalwari (CT),124692,118,...,42915,KALWARI,766,BICHPURI,00807,"Kalwari (CT) , Agra,Agra,UTTAR PRADESH",7572.232960,3.233625e+06,21-146-00807,"POLYGON ((77.94733 27.18884, 77.94392 27.18546..."
1,DHANAULI,09,146,00766,UTTAR PRADESH,Agra,Agra,Dhanauli (CT),124698,118,...,241772,DHANAULI,766,BICHPURI,00807,"Dhanauli (CT) , Agra,Agra,UTTAR PRADESH",13227.480482,9.285302e+06,21-146-00807,"POLYGON ((77.95845 27.15603, 77.95237 27.15677..."
2,AZIZPUR,09,146,00766,UTTAR PRADESH,Agra,Agra,Azizpur (CT),124697,118,...,274049,AZIZPUR,766,BICHPURI,00807,"Azizpur (CT) , Agra,Agra,UTTAR PRADESH",6170.711891,1.808432e+06,21-146-00807,"POLYGON ((77.97839 27.13672, 77.98048 27.13899..."
3,BESRAMPUR,09,146,00768,UTTAR PRADESH,Agra,Kheragarh,Bishrampur,124945,118,...,43198,BISHRAMPUR,768,KHERAGARH,00814,"Bishrampur , Kheragarh,Agra,UTTAR PRADESH",7137.784996,2.408859e+06,21-146-00814,"POLYGON ((77.87975 27.01265, 77.87692 27.01329..."
4,SIKRI,09,146,00767,UTTAR PRADESH,Agra,Kiraoli,Fatehpur Sikri (NPP),800808,118,...,0,None,767,FATEHPUR SIKRI,00810,"Fatehpur Sikri (NPP) , Kiraoli,Agra,UTTAR PRADESH",19001.820920,1.375611e+07,21-146-00810,"POLYGON ((77.66733 27.11214, 77.65673 27.11386..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1612,KOTWA,09,197,00996,UTTAR PRADESH,Varanasi,Sadar,Kotwa (CT),209755,187,...,243378,KOTAWA,996,KASHI VIDYAPEETH,01621,"Kotwa (CT) , Sadar,Varanasi,UTTAR PRADESH",9054.014162,2.770890e+06,21-197-01621,"POLYGON ((82.93605 25.34918, 82.92451 25.35091..."
1613,CHHITAWNI,09,197,00996,UTTAR PRADESH,Varanasi,Sadar,Chhitauni (CT),209735,187,...,93402,CHITOUNI KOT,996,KASHI VIDYAPEETH,01621,"Chhitauni (CT) , Sadar,Varanasi,UTTAR PRADESH",6991.195967,2.765621e+06,21-197-01621,"POLYGON ((82.94027 25.33132, 82.93544 25.34088..."
1614,KORUTT,09,197,00996,UTTAR PRADESH,Varanasi,Sadar,Kotwa (CT),209752,187,...,0,None,996,ARAJILINE,01616,"Kotwa (CT) , Sadar,Varanasi,UTTAR PRADESH",17753.005254,1.172267e+07,21-197-01616,"POLYGON ((82.89916 25.36802, 82.89546 25.36861..."
1615,DEORAI,09,197,00995,UTTAR PRADESH,Varanasi,Pindra,Deorai,208866,187,...,93298,BANTARI,995,CHOLAPUR,01619,"Deorai , Pindra,Varanasi,UTTAR PRADESH",2644.722887,3.556189e+05,21-197-01619,"POLYGON ((82.98421 25.55578, 82.98088 25.55657..."


In [4]:
od_urban_shapes = od_urban_shapes[~(od_urban_shapes['geometry'].is_empty | od_urban_shapes['geometry'].isna())]


In [8]:
# TOTAL POPULATION IN EACH URBAN AREA
import rasterio
import rasterstats

worldpop_raster = rasterio.open(r'D:\CDL\flood-data-ecosystem-UP\Sources\WORLDPOP\data\population_counts\UP_ppp_2020.tif')
worldpop_raster_array = worldpop_raster.read(1)

sum_dicts = rasterstats.zonal_stats(od_urban_shapes.to_crs(worldpop_raster.crs),
                                     worldpop_raster_array,
                                     affine= worldpop_raster.transform,
                                     stats= ['sum'],
                                     nodata=worldpop_raster.nodata,
                                     geojson_out = True)

dfs = []
for sd in sum_dicts:
    dfs.append(pd.DataFrame([sd['properties']]))

pop_zonal_stats_df = pd.concat(dfs).reset_index(drop=True)
pop_zonal_stats_df = pop_zonal_stats_df.rename(columns={'sum':'sum_population'})

C:\Users\cdl\AppData\Local\Temp\ipykernel_1260\204467640.py:19: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  pop_zonal_stats_df = pd.concat(dfs).reset_index(drop=True)


In [ ]:
# Divide population by 4.6 for households : source census 2011

pop_zonal_stats_df['urban_hhd'] = pop_zonal_stats_df['sum_population']/4.6

In [11]:
urban_hhd = pop_zonal_stats_df.groupby('sdtcode11')[['urban_hhd']].sum().reset_index()
urban_hhd

,sdtcode11,urban_hhd
0,00701,7042.185211
1,00702,122264.804475
2,00703,15809.854206
3,00704,12212.239035
4,00705,10024.471024
...,...,...
302,01009,10518.399287
303,01010,20661.298483
304,01011,4017.272418
305,01012,14250.348643


In [12]:
urban_hhd['urban_electricity'] = urban_hhd['urban_hhd']*20

In [13]:
urban_hhd['urban_tele'] = urban_hhd['urban_hhd']*3

In [15]:
urban_hhd['urban_hhd_pipe'] = urban_hhd['urban_hhd']*0.1873

In [16]:
urban_hhd['urban_no_sanitation'] = urban_hhd['urban_hhd']*0.063

In [17]:
urban_hhd = urban_hhd.rename(columns={'sdtcode11': 'sub_district_code'})

In [18]:
urban_hhd['sub_district_code'] = urban_hhd['sub_district_code'].apply(lambda x: x[1:] if x.startswith('0') else x)
urban_hhd

,sub_district_code,urban_hhd,urban_electricity,urban_tele,urban_hhd_pipe,urban_no_sanitation
0,0701,7042.185211,1.408437e+05,21126.555634,1319.001290,443.657668
1,0702,122264.804475,2.445296e+06,366794.413426,22900.197878,7702.682682
2,0703,15809.854206,3.161971e+05,47429.562617,2961.185693,996.020815
3,0704,12212.239035,2.442448e+05,36636.717105,2287.352371,769.371059
4,0705,10024.471024,2.004894e+05,30073.413073,1877.583423,631.541675
...,...,...,...,...,...,...
302,1009,10518.399287,2.103680e+05,31555.197860,1970.096186,662.659155
303,1010,20661.298483,4.132260e+05,61983.895449,3869.861206,1301.661804
304,1011,4017.272418,8.034545e+04,12051.817255,752.435124,253.088162
305,1012,14250.348643,2.850070e+05,42751.045930,2669.090301,897.771965


# Vul variables

In [20]:
vul_df = pd.read_csv(r'D:\CDL\flood-data-ecosystem-UP\Sources\ANTYODAYA\data\UP_ANTYODAYA.csv')

In [21]:
vul_df = vul_df.rename(columns={'total_hhd':'rural_hhd'})
vul_df['sub_district_code'] = vul_df['sub_district_code'].astype(str)


## net_sown_area_in_hac

In [72]:
#original 
net_sown_area_in_hac_sd = vul_df.groupby(['sub_district_code'])

In [73]:
net_sown_area_in_hac_sd = vul_df.groupby(['sub_district_code'])[['net_sown_area_in_hac']].sum()

KeyError: "Columns not found: 'net_sown_area_in_hac'"

In [74]:
net_sown_area_in_hac_sd

## Rural - Urban Households in each rc

In [22]:
rural_hhd = vul_df.groupby('sub_district_code')[['rural_hhd']].sum().reset_index()
rural_hhd

,sub_district_code,rural_hhd
0,1000,236829.0
1,1001,113441.0
2,1002,70327.0
3,1003,189563.0
4,1004,76814.0
...,...,...
345,995,138443.0
346,996,231027.0
347,997,177420.0
348,998,154597.0


In [25]:
vul_df[vul_df.sub_district_code == 6369]

,gp_code,village_code,sub_district_code,sub_district_name,rural_hhd,availablility_hours_of_domestic_electricity,availability_of_telephone_services,total_hhd_having_piped_water_connection,total_hhd_not_having_sanitary_latrines


In [26]:
#urban_hhd['sub_district_code'] = urban_hhd['sub_district_code'].astype(str)
total_hhd = urban_hhd.merge(rural_hhd, how='outer', on='sub_district_code')
total_hhd

,sub_district_code,urban_hhd,urban_electricity,urban_tele,urban_hhd_pipe,urban_no_sanitation,rural_hhd
0,0701,7042.185211,1.408437e+05,21126.555634,1319.001290,443.657668,NaN
1,0702,122264.804475,2.445296e+06,366794.413426,22900.197878,7702.682682,NaN
2,0703,15809.854206,3.161971e+05,47429.562617,2961.185693,996.020815,NaN
3,0704,12212.239035,2.442448e+05,36636.717105,2287.352371,769.371059,NaN
4,0705,10024.471024,2.004894e+05,30073.413073,1877.583423,631.541675,NaN
...,...,...,...,...,...,...,...
641,995,NaN,NaN,NaN,NaN,NaN,138443.0
642,996,NaN,NaN,NaN,NaN,NaN,231027.0
643,997,NaN,NaN,NaN,NaN,NaN,177420.0
644,998,NaN,NaN,NaN,NaN,NaN,154597.0


In [27]:
total_hhd = total_hhd.fillna(0)
total_hhd['total_hhd']=total_hhd['rural_hhd']+total_hhd['urban_hhd']

In [28]:
total_hhd

,sub_district_code,urban_hhd,urban_electricity,urban_tele,urban_hhd_pipe,urban_no_sanitation,rural_hhd,total_hhd
0,0701,7042.185211,1.408437e+05,21126.555634,1319.001290,443.657668,0.0,7042.185211
1,0702,122264.804475,2.445296e+06,366794.413426,22900.197878,7702.682682,0.0,122264.804475
2,0703,15809.854206,3.161971e+05,47429.562617,2961.185693,996.020815,0.0,15809.854206
3,0704,12212.239035,2.442448e+05,36636.717105,2287.352371,769.371059,0.0,12212.239035
4,0705,10024.471024,2.004894e+05,30073.413073,1877.583423,631.541675,0.0,10024.471024
...,...,...,...,...,...,...,...,...
641,995,0.000000,0.000000e+00,0.000000,0.000000,0.000000,138443.0,138443.000000
642,996,0.000000,0.000000e+00,0.000000,0.000000,0.000000,231027.0,231027.000000
643,997,0.000000,0.000000e+00,0.000000,0.000000,0.000000,177420.0,177420.000000
644,998,0.000000,0.000000e+00,0.000000,0.000000,0.000000,154597.0,154597.000000


## availablility_hours_of_domestic_electricity

In [29]:
# Map category to mean electricity available
electricity_dict = {1:2.5, 2:6, 3:10, 4:12, 5:0}

In [30]:
elect_df = vul_df.replace({"availablility_hours_of_domestic_electricity": electricity_dict})[['sub_district_code','rural_hhd','availablility_hours_of_domestic_electricity']]

In [31]:
elect_df['rural_electricity'] = elect_df['rural_hhd']*elect_df['availablility_hours_of_domestic_electricity']

In [33]:
elect_df = elect_df.groupby('sub_district_code')[['rural_hhd','rural_electricity']].sum().reset_index()

In [34]:
elect_df

,sub_district_code,rural_hhd,rural_electricity
0,1000,236829.0,2579295.5
1,1001,113441.0,1061428.0
2,1002,70327.0,710736.5
3,1003,189563.0,1971423.5
4,1004,76814.0,795929.5
...,...,...,...
345,995,138443.0,1618120.0
346,996,231027.0,2733672.0
347,997,177420.0,1812154.5
348,998,154597.0,1622832.0


In [35]:
elect_df['sub_district_code'] = elect_df['sub_district_code'].astype(str)
sd_electricity = elect_df.merge(urban_hhd, on='sub_district_code', how='outer')

In [36]:
sd_electricity = sd_electricity.fillna(0)

In [37]:
sd_electricity['total_electricity'] = sd_electricity['rural_electricity'] + sd_electricity['urban_electricity']

In [38]:
sd_electricity['total_hhd'] = sd_electricity['rural_hhd'] + sd_electricity['urban_hhd']

In [39]:
sd_electricity

,sub_district_code,rural_hhd,rural_electricity,urban_hhd,urban_electricity,urban_tele,urban_hhd_pipe,urban_no_sanitation,total_electricity,total_hhd
0,0701,0.0,0.0,7042.185211,1.408437e+05,21126.555634,1319.001290,443.657668,1.408437e+05,7042.185211
1,0702,0.0,0.0,122264.804475,2.445296e+06,366794.413426,22900.197878,7702.682682,2.445296e+06,122264.804475
2,0703,0.0,0.0,15809.854206,3.161971e+05,47429.562617,2961.185693,996.020815,3.161971e+05,15809.854206
3,0704,0.0,0.0,12212.239035,2.442448e+05,36636.717105,2287.352371,769.371059,2.442448e+05,12212.239035
4,0705,0.0,0.0,10024.471024,2.004894e+05,30073.413073,1877.583423,631.541675,2.004894e+05,10024.471024
...,...,...,...,...,...,...,...,...,...,...
641,995,138443.0,1618120.0,0.000000,0.000000e+00,0.000000,0.000000,0.000000,1.618120e+06,138443.000000
642,996,231027.0,2733672.0,0.000000,0.000000e+00,0.000000,0.000000,0.000000,2.733672e+06,231027.000000
643,997,177420.0,1812154.5,0.000000,0.000000e+00,0.000000,0.000000,0.000000,1.812154e+06,177420.000000
644,998,154597.0,1622832.0,0.000000,0.000000e+00,0.000000,0.000000,0.000000,1.622832e+06,154597.000000


In [40]:
sd_electricity['avg_electricity'] = sd_electricity['total_electricity']/sd_electricity['total_hhd']

In [41]:
sd_electricity

,sub_district_code,rural_hhd,rural_electricity,urban_hhd,urban_electricity,urban_tele,urban_hhd_pipe,urban_no_sanitation,total_electricity,total_hhd,avg_electricity
0,0701,0.0,0.0,7042.185211,1.408437e+05,21126.555634,1319.001290,443.657668,1.408437e+05,7042.185211,20.000000
1,0702,0.0,0.0,122264.804475,2.445296e+06,366794.413426,22900.197878,7702.682682,2.445296e+06,122264.804475,20.000000
2,0703,0.0,0.0,15809.854206,3.161971e+05,47429.562617,2961.185693,996.020815,3.161971e+05,15809.854206,20.000000
3,0704,0.0,0.0,12212.239035,2.442448e+05,36636.717105,2287.352371,769.371059,2.442448e+05,12212.239035,20.000000
4,0705,0.0,0.0,10024.471024,2.004894e+05,30073.413073,1877.583423,631.541675,2.004894e+05,10024.471024,20.000000
...,...,...,...,...,...,...,...,...,...,...,...
641,995,138443.0,1618120.0,0.000000,0.000000e+00,0.000000,0.000000,0.000000,1.618120e+06,138443.000000,11.687987
642,996,231027.0,2733672.0,0.000000,0.000000e+00,0.000000,0.000000,0.000000,2.733672e+06,231027.000000,11.832695
643,997,177420.0,1812154.5,0.000000,0.000000e+00,0.000000,0.000000,0.000000,1.812154e+06,177420.000000,10.213925
644,998,154597.0,1622832.0,0.000000,0.000000e+00,0.000000,0.000000,0.000000,1.622832e+06,154597.000000,10.497177


## availability_of_telephone_services

In [42]:
telephone_df = vul_df.copy()

In [43]:
telephone_df['rural_tele'] = vul_df['rural_hhd']*vul_df['availability_of_telephone_services']

In [44]:
telephone_df = telephone_df[['sub_district_code', 'rural_hhd', 'rural_tele']]

In [45]:
telephone_df = telephone_df.groupby('sub_district_code')[['rural_hhd','rural_tele']].sum().reset_index()

In [46]:
#dependent on urban_shapes
#rural_hhd['sub_district_code'] = rural_hhd['sub_district_code'].astype(str)
sd_tele = telephone_df.merge(urban_hhd[['sub_district_code', 'urban_hhd','urban_tele']], on='sub_district_code', how='outer')

In [47]:
sd_tele = sd_tele.fillna(0)

In [49]:
sd_tele['total_tele'] = sd_tele['rural_tele'] + sd_tele['urban_tele']

In [50]:
sd_tele['total_hhd'] = sd_tele['rural_hhd'] + sd_tele['urban_hhd']

In [51]:
sd_tele['avg_tele'] = round(sd_tele['total_tele']/sd_tele['total_hhd'])

In [52]:
sd_tele

,sub_district_code,rural_hhd,rural_tele,urban_hhd,urban_tele,total_tele,total_hhd,avg_tele
0,0701,0.0,0.0,7042.185211,21126.555634,21126.555634,7042.185211,3.0
1,0702,0.0,0.0,122264.804475,366794.413426,366794.413426,122264.804475,3.0
2,0703,0.0,0.0,15809.854206,47429.562617,47429.562617,15809.854206,3.0
3,0704,0.0,0.0,12212.239035,36636.717105,36636.717105,12212.239035,3.0
4,0705,0.0,0.0,10024.471024,30073.413073,30073.413073,10024.471024,3.0
...,...,...,...,...,...,...,...,...
641,995,138443.0,321635.0,0.000000,0.000000,321635.000000,138443.000000,2.0
642,996,231027.0,556480.0,0.000000,0.000000,556480.000000,231027.000000,2.0
643,997,177420.0,389577.0,0.000000,0.000000,389577.000000,177420.000000,2.0
644,998,154597.0,344024.0,0.000000,0.000000,344024.000000,154597.000000,2.0


## total_hhd_having_piped_water_connection

In [53]:
pipe_df = vul_df.copy()

In [54]:
pipe_df = pipe_df.groupby('sub_district_code')[['total_hhd_having_piped_water_connection']].sum().reset_index()

In [55]:
pipe_df = pipe_df.merge(urban_hhd[['sub_district_code','urban_hhd_pipe']], on='sub_district_code', how='outer')

In [56]:
pipe_df = pipe_df.fillna(0)

In [57]:
pipe_df['sd_piped_hhds'] = pipe_df['urban_hhd_pipe'] + pipe_df['total_hhd_having_piped_water_connection']

In [58]:
pipe_df = pipe_df.merge(total_hhd[['sub_district_code','total_hhd']], on='sub_district_code')

In [59]:
pipe_df['sd_piped_hhds_pct'] = 100*pipe_df['sd_piped_hhds']/pipe_df['total_hhd']

In [60]:
pipe_df

,sub_district_code,total_hhd_having_piped_water_connection,urban_hhd_pipe,sd_piped_hhds,total_hhd,sd_piped_hhds_pct
0,0701,0.0,1319.001290,1319.001290,7042.185211,18.730000
1,0702,0.0,22900.197878,22900.197878,122264.804475,18.730000
2,0703,0.0,2961.185693,2961.185693,15809.854206,18.730000
3,0704,0.0,2287.352371,2287.352371,12212.239035,18.730000
4,0705,0.0,1877.583423,1877.583423,10024.471024,18.730000
...,...,...,...,...,...,...
641,995,20337.0,0.000000,20337.000000,138443.000000,14.689800
642,996,44764.0,0.000000,44764.000000,231027.000000,19.376090
643,997,12120.0,0.000000,12120.000000,177420.000000,6.831248
644,998,22588.0,0.000000,22588.000000,154597.000000,14.610892


## total_hhd_not_having_sanitary_latrines

In [61]:
nosanitation_df = vul_df.copy()

In [62]:
nosanitation_df = nosanitation_df.groupby('sub_district_code')[['total_hhd_not_having_sanitary_latrines']].sum().reset_index()

In [63]:
nosanitation_df = nosanitation_df.merge(urban_hhd[['sub_district_code','urban_no_sanitation']], on='sub_district_code', how='outer')

In [64]:
nosanitation_df = nosanitation_df.fillna(0)

In [65]:
nosanitation_df['sd_nosanitation_hhds'] = nosanitation_df['urban_no_sanitation'] + nosanitation_df['total_hhd_not_having_sanitary_latrines']

In [66]:
nosanitation_df = nosanitation_df.merge(total_hhd[['sub_district_code','total_hhd']], on='sub_district_code')

In [67]:
nosanitation_df['sd_nosanitation_hhds_pct'] = 100*nosanitation_df['sd_nosanitation_hhds']/nosanitation_df['total_hhd']

In [68]:
nosanitation_df

,sub_district_code,total_hhd_not_having_sanitary_latrines,urban_no_sanitation,sd_nosanitation_hhds,total_hhd,sd_nosanitation_hhds_pct
0,0701,0.0,443.657668,443.657668,7042.185211,6.300000
1,0702,0.0,7702.682682,7702.682682,122264.804475,6.300000
2,0703,0.0,996.020815,996.020815,15809.854206,6.300000
3,0704,0.0,769.371059,769.371059,12212.239035,6.300000
4,0705,0.0,631.541675,631.541675,10024.471024,6.300000
...,...,...,...,...,...,...
641,995,7659.0,0.000000,7659.000000,138443.000000,5.532241
642,996,7687.0,0.000000,7687.000000,231027.000000,3.327317
643,997,19806.0,0.000000,19806.000000,177420.000000,11.163341
644,998,12298.0,0.000000,12298.000000,154597.000000,7.954876


# ANTYODAYA MASTER

In [83]:
antyodaya_master_df = total_hhd



In [84]:
antyodaya_master_df = antyodaya_master_df.merge(sd_electricity[['sub_district_code', 'avg_electricity']], how='outer')

In [85]:
antyodaya_master_df = antyodaya_master_df.merge(sd_tele[['sub_district_code', 'avg_tele']])

In [86]:
antyodaya_master_df = antyodaya_master_df.merge(pipe_df[['sub_district_code', 'sd_piped_hhds_pct']])

In [87]:
antyodaya_master_df = antyodaya_master_df.merge(nosanitation_df[['sub_district_code','sd_nosanitation_hhds_pct']])


In [88]:
antyodaya_master_df

,sub_district_code,urban_hhd,urban_electricity,urban_tele,urban_hhd_pipe,urban_no_sanitation,rural_hhd,total_hhd,avg_electricity,avg_tele,sd_piped_hhds_pct,sd_nosanitation_hhds_pct
0,0701,7042.185211,1.408437e+05,21126.555634,1319.001290,443.657668,0.0,7042.185211,20.000000,3.0,18.730000,6.300000
1,0702,122264.804475,2.445296e+06,366794.413426,22900.197878,7702.682682,0.0,122264.804475,20.000000,3.0,18.730000,6.300000
2,0703,15809.854206,3.161971e+05,47429.562617,2961.185693,996.020815,0.0,15809.854206,20.000000,3.0,18.730000,6.300000
3,0704,12212.239035,2.442448e+05,36636.717105,2287.352371,769.371059,0.0,12212.239035,20.000000,3.0,18.730000,6.300000
4,0705,10024.471024,2.004894e+05,30073.413073,1877.583423,631.541675,0.0,10024.471024,20.000000,3.0,18.730000,6.300000
...,...,...,...,...,...,...,...,...,...,...,...,...
641,995,0.000000,0.000000e+00,0.000000,0.000000,0.000000,138443.0,138443.000000,11.687987,2.0,14.689800,5.532241
642,996,0.000000,0.000000e+00,0.000000,0.000000,0.000000,231027.0,231027.000000,11.832695,2.0,19.376090,3.327317
643,997,0.000000,0.000000e+00,0.000000,0.000000,0.000000,177420.0,177420.000000,10.213925,2.0,6.831248,11.163341
644,998,0.000000,0.000000e+00,0.000000,0.000000,0.000000,154597.0,154597.000000,10.497177,2.0,14.610892,7.954876


In [113]:
gdf_polygons
print(gdf_polygons.crs)
gdf_polygons = gdf_polygons.to_crs('EPSG:4326')
gdf_polygons



AttributeError: 'NoneType' object has no attribute 'crs'

In [147]:
antyodaya_master_df.head()

,objectid,stcode11,dtcode11,sub_district_code,stname,dtname,sdtname,st_area_sh,st_length_,remarks,...,urban_tele,urban_hhd_pipe,urban_no_sanitation,rural_hhd,total_hhd_x,avg_electricity,avg_tele,sd_piped_hhds_pct,sd_nosanitation_hhds_pct,total_hhd_y
0,09-132-00701,09,132,0701,UTTAR PRADESH,Saharanpur,Behat,0.0,0.0,,...,21126.555634,1319.001290,443.657668,0.0,7042.185211,20.0,3.0,18.73,6.3,7042.185211
1,09-132-00702,09,132,0702,UTTAR PRADESH,Saharanpur,Saharanpur,0.0,0.0,,...,366794.413426,22900.197878,7702.682682,0.0,122264.804475,20.0,3.0,18.73,6.3,122264.804475
2,09-132-00703,09,132,0703,UTTAR PRADESH,Saharanpur,Nakur,0.0,0.0,,...,47429.562617,2961.185693,996.020815,0.0,15809.854206,20.0,3.0,18.73,6.3,15809.854206
3,09-132-00704,09,132,0704,UTTAR PRADESH,Saharanpur,Deoband,0.0,0.0,,...,36636.717105,2287.352371,769.371059,0.0,12212.239035,20.0,3.0,18.73,6.3,12212.239035
4,09-132-00705,09,132,0705,UTTAR PRADESH,Saharanpur,Rampur Maniharan,0.0,0.0,,...,30073.413073,1877.583423,631.541675,0.0,10024.471024,20.0,3.0,18.73,6.3,10024.471024


In [148]:

antyodaya_master_df = pd.merge(gdf_polygons_renamed,antyodaya_master_df,on='sub_district_code', how='outer')

In [149]:
antyodaya_master_df[antyodaya_master_df.avg_electricity.isnull()]

,objectid_x,stcode11_x,dtcode11_x,sub_district_code,stname_x,dtname_x,sdtname_x,st_area_sh_x,st_length__x,remarks_x,...,urban_tele,urban_hhd_pipe,urban_no_sanitation,rural_hhd,total_hhd_x,avg_electricity,avg_tele,sd_piped_hhds_pct,sd_nosanitation_hhds_pct,total_hhd_y
145,09-161-00837,09,161,0837,UTTAR PRADESH,Etawah,Saifai,0.0,0.0,,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
228,09-180-00920,09,180,0920,UTTAR PRADESH,Bahraich,Mahasi,0.0,0.0,,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
287,09-193-00980,09,193,0980,UTTAR PRADESH,Ballia,Bairia,0.0,0.0,,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
#antyodaya_master_df2 = antyodaya_master_df.merge(total_hhd[['sub_district_code', 'total_hhd']], on='sub_district_code')
antyodaya_master_df['total_hhd'] = antyodaya_master_df['total_hhd_y'].apply(lambda x: round(x,0))

In [ ]:
antyodaya_master_df = antyodaya_master_df.drop(['geometry_x', 'geometry_y'],axis=1)


In [ ]:
antyodaya_master_df.to_csv(r'D:\CDL\flood-data-ecosystem-UP\Sources\ANTYODAYA\data\antyodaya_variables.csv', index=False)


In [ ]:
#original code for reference
'''antyodaya_master_df.drop(['geometry',
                           'district_1', 'revenue_cr', 'HQ', 'area', 
                           'are_new'],axis=1).to_csv('/home/krishna/IDS-DRR-Data-Pipeline/Sources/ANTYODAYA/data/variables/antyodaya/antyodaya_variables.csv', index=False)
'''

"antyodaya_master_df.drop(['geometry',\n                           'district_1', 'revenue_cr', 'HQ', 'area', \n                           'are_new'],axis=1).to_csv('/home/krishna/IDS-DRR-Data-Pipeline/Sources/ANTYODAYA/data/variables/antyodaya/antyodaya_variables.csv', index=False)\n"

In [162]:
antyodaya_master_df

,sub_district_code,dtname_x,sdtname_x,remarks_x,dist_lgd_x,state_lgd_x,subdt_lgd_x,ac_no_x,st_length(shape)_x,st_area(shape)_x,...,urban_hhd_pipe,urban_no_sanitation,rural_hhd,total_hhd_x,avg_electricity,avg_tele,sd_piped_hhds_pct,sd_nosanitation_hhds_pct,total_hhd_y,total_hhd
0,0701,Saharanpur,Behat,,177.0,9.0,701.0,1.0,209734.003509,1.378660e+09,...,1319.001290,443.657668,0.0,7042.185211,20.000000,3.0,18.730000,6.300000,7042.185211,7042.0
1,0702,Saharanpur,Saharanpur,,177.0,9.0,702.0,3.0,238816.450116,1.058847e+09,...,22900.197878,7702.682682,0.0,122264.804475,20.000000,3.0,18.730000,6.300000,122264.804475,122265.0
2,0703,Saharanpur,Nakur,,177.0,9.0,703.0,7.0,208398.959818,1.157326e+09,...,2961.185693,996.020815,0.0,15809.854206,20.000000,3.0,18.730000,6.300000,15809.854206,15810.0
3,0704,Saharanpur,Deoband,,177.0,9.0,704.0,5.0,157617.530216,7.409517e+08,...,2287.352371,769.371059,0.0,12212.239035,20.000000,3.0,18.730000,6.300000,12212.239035,12212.0
4,0705,Saharanpur,Rampur Maniharan,,177.0,9.0,705.0,7.0,150206.795681,6.318282e+08,...,1877.583423,631.541675,0.0,10024.471024,20.000000,3.0,18.730000,6.300000,10024.471024,10024.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
653,995,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.000000,0.000000,138443.0,138443.000000,11.687987,2.0,14.689800,5.532241,138443.000000,138443.0
654,996,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.000000,0.000000,231027.0,231027.000000,11.832695,2.0,19.376090,3.327317,231027.000000,231027.0
655,997,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.000000,0.000000,177420.0,177420.000000,10.213925,2.0,6.831248,11.163341,177420.000000,177420.0
656,998,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.000000,0.000000,154597.0,154597.000000,10.497177,2.0,14.610892,7.954876,154597.000000,154597.0
